# Análise de Dados de Recursos Humanos

**Instituto FIAP** | **Professor:** Luciana | **Turma:** 2TDSPH

---

## Apresentação do Projeto

Este projeto consiste em uma análise exploratória de dados de Recursos Humanos, utilizando um dataset contendo informações de **311 funcionários** e **36 colunas**. O objetivo é aplicar técnicas de programação Python e manipulação de dados com Pandas para extrair insights relevantes sobre a força de trabalho.

### Estrutura do Notebook

Este notebook está organizado em **8 exercícios indexados**, progressivos em complexidade:

| Exercício | Tópico |
|-----------|-------|
| **Exercício 1** | Reconhecimento estrutural e inspeção da base de dados |
| **Exercício 2** | Lógica de classificação com if/elif/else e funções |
| **Exercício 3** | Uso de funções lambda e filtragem com Pandas |
| **Exercício 4** | Análises agregadas e ranking |
| **Exercício 5** | Classificação sofisticada com múltiplas variáveis |
| **Exercício 6** | Análise de amostragem e variabilidade |
| **Exercício 7** | Identificação de padrões e achados |
| **Exercício 8** | Conclusão |

---

## Base de Dados: HRDataset_v14.csv

### Descrição

O dataset utilizado é o **HRDataset_v14**, contendo dados de funcionários de uma empresa. Cada linha representa um сотрудник (funcionário) e cada coluna representa um atributo.

### Estrutura

| Característica | Valor |
|----------------|-------|
| Total de registros | 311 |
| Total de colunas | 36 |

### Principais Colunas

| Coluna | Descrição |
|--------|----------|
| `Employee_Name` | Nome do funcionário |
| `EmpID` | ID do funcionário |
| `Salary` | Salário |
| `Department` | Departamento |
| `Position` | Cargo |
| `EmploymentStatus` | Status de employment (Active/Inactive) |
| `ManagerID` | ID do gerente |
| `PerformanceScore` | Pontuação de desempenho (Exceeds, Fully Meets, Needs Improvement, PIP) |
| `EngagementSurvey` | Pesquisa de engajamento (0-5) |
| `EmpSatisfaction` | Satisfação do funcionário (1-5) |
| `Absences` | Número de ausências |
| `DaysLateLast30` | Dias de atraso nos últimos 30 dias |

### Observações sobre Dados

- **Valores nulos:** `ManagerID` possui alguns valores ausentes (8 registros)
- **Duplicatas:** Não há linhas duplicadas
- **Status:** `DateofTermination` possui valores nulos para funcionários ativos (esperado)

---


## Exercício 1: Reconhecimento Estrutural e Inspeção da Base de Dados

**Objetivo:** Entender a estrutura básica do DataFrame — dimensões, tipos de dados, valores ausentes e duplicados.

In [ ]:
import pandas as pd
import numpy as np

# Carrega os dados
df = pd.read_csv('HRDataset_v14.csv')

# Dimensões
print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")

# Lista de colunas
print("\nColunas:")
for col in df.columns:
    print(f"  - {col}")

In [ ]:
# Tipos de dados e informações gerais
df.info()

In [ ]:
# Valores ausentes
print("Valores ausentes por coluna:")
print(df.isnull().sum())

print(f"\nLinhas duplicadas: {df.duplicated().sum()}")

### Tratamento de Valores Nulos

A coluna `ManagerID` possui valores ausentes. Optamos por preencher com a **moda** (valor mais frequente) para manter a semântica de um gerente existente na base.

In [ ]:
# Preenche ManagerID com a moda
manager_id_mode = df['ManagerID'].mode()[0]
df['ManagerID'].fillna(manager_id_mode, inplace=True)
df['ManagerID'] = df['ManagerID'].astype(int)

# Confirma tratamento
print(f"Valores nulos em ManagerID após tratamento: {df['ManagerID'].isnull().sum()}")

---

## Exercício 2: Lógica de Classificação com Funções

**Objetivo:** Criar funções de classificação e aplicá-las ao DataFrame usando `.apply()`.

In [ ]:
# Função 1: Classifica salário em Baixo, Médio ou Alto
def classificar_salario(salario):
    q1 = df['Salary'].quantile(0.33)
    q3 = df['Salary'].quantile(0.66)
    
    if salario < q1:
        return 'Baixo'
    elif salario < q3:
        return 'Médio'
    else:
        return 'Alto'

# Função 2: Interpreta pontuação de desempenho
def interpretar_desempenho(score):
    if score == 'Exceeds':
        return 'Acima do Esperado'
    elif score == 'Fully Meets':
        return 'Conforme o Esperado'
    elif score == 'Needs Improvement':
        return 'Precisa de Melhorias'
    elif score == 'PIP':
        return 'Em Plano de Melhoria'
    else:
        return 'Não Avaliado'

# Aplica as funções
df['Categoria_Salario'] = df['Salary'].apply(classificar_salario)
df['Descricao_Performance'] = df['PerformanceScore'].apply(interpretar_desempenho)

# Resultado
print("Classificação Salarial:")
print(df['Categoria_Salario'].value_counts())

print("\nDesempenho dos Funcionários:")
print(df['Descricao_Performance'].value_counts())

---

## Exercício 3: Funções Lambda e Filtragem

**Objetivo:** Criar novas colunas com funções lambda e filtrar dados por condições específicas.

In [ ]:
# Lambda: converte nome para maiúsculas
df['Nome_Upper'] = df['Employee_Name'].apply(lambda x: x.upper())

# Filtra apenas funcionários ativos
df_ativos = df[df['EmploymentStatus'] == 'Active'].copy()

print(f"Total de funcionários: {len(df)}")
print(f"Funcionários ativos: {len(df_ativos)}")

df_ativos[['Employee_Name', 'Nome_Upper', 'EmploymentStatus']].head()

**Por que filtrar apenas ativos?** Permite focar na força de trabalho atual, evitando distorções caused by dados de ex-funcionários.

---

## Exercício 4: Análises Agregadas e Ranking

**Objetivo:** Agrupar dados por categoria e calcular métricas agregadas para criar rankings.

In [ ]:
# Média de salário por departamento
ranking_departamentos = df.groupby('Department')['Salary'].mean().sort_values(ascending=False)

print("Ranking de Salário Médio por Departamento:\n")
ranking_departamentos.head(10)

**Insights:**
- `Executive Office` possui o maior salário médio
- Áreas de tecnologia (`Software Engineering`, `IT/IS`) aparecem no topo
- Reflete a demanda de mercado e especialização técnica

---

## Exercício 5: Classificação com Múltiplas Variáveis

**Objetivo:** Combinar variáveis para criar scores e classificar funcionários em categorias estratégicas.

In [ ]:
# Cria score combinando salário e engajamento
df['Score_Analise'] = df['Salary'] * df['EngagementSurvey']

# Classifica em 3 categorias usando np.where
q1 = df['Score_Analise'].quantile(0.33)
q2 = df['Score_Analise'].quantile(0.66)

df['Classificacao_Final'] = np.where(
    df['Score_Analise'] < q1, 'Atenção',
    np.where(df['Score_Analise'] < q2, 'Regular', 'Excelente')
)

print("Distribuição dos Funcionários por Categoria:\n")
print(df['Classificacao_Final'].value_counts())

### Interpretação das Categorias

| Categoria | Significado | Ação Recomendada |
|-----------|------------|-----------------|
| **Excelente** | Alto salário + alto engajamento | Programas de retenção, sucessão |
| **Regular** | Média salarial e engajamento | Treinamentos, feedbacks |
| **Atenção** | Baixo score | Intervenção imediata, mentoria |

---

## Exercício 6: Análise de Amostragem e Variabilidade

**Objetivo:** Compreender conceitos de representatividade e variabilidade amostral.

In [ ]:
# Gera amostra aleatória (20% dos dados)
df_amostra = df.sample(frac=0.2, random_state=42)

print(f"Base completa: {len(df)} registros")
print(f"Amostra (20%): {len(df_amostra)} registros")

# Compara média do salário
print(f"\nMédia salário (base completa): R$ {df['Salary'].mean():,.2f}")
print(f"Média salário (amostra): R$ {df_amostra['Salary'].mean():,.2f}")

In [ ]:
# Variabilidade: múltiplas amostras
print("Médias de 5 amostras diferentes (20% cada):\n")
for i in range(1, 6):
    amostra = df.sample(frac=0.2, random_state=i)
    print(f"  Amostra {i}: R$ {amostra['Salary'].mean():,.2f}")

### Conclusões sobre Amostragem

- **Representatividade:** A média da amostra está próxima da média da população
- **Variabilidade:** Diferentes amostras geram médias ligeiramente diferentes
- Este fenômeno é natural e crucial para inferências estatísticas

---

## Exercício 7: Identificação de Padrões e Achados

### Principais Descobertas

1. **Estrutura Salarial por Departamento**
   - `Executive Office`, `IT/IS` e `Software Engineering` têm os maiores salários
   - Reflete valorização de liderança e especialização técnica

2. **Segmentação da Força de Trabalho**
   - Combinar `Salary` × `EngagementSurvey` criou metric powerful
   - Categorias permitem ações direcionadas de RH

3. **Qualidade dos Dados**
   - Tratamento de valores ausentes em `ManagerID` foi essencial
   - Filtragem por `EmploymentStatus` evitou distorções

---

## Exercício 8: Conclusão

### Pergunta Central

*Como a análise exploratória e a segmentação de dados de RH podem informar estratégias eficazes de gestão de talentos e compensação?*

### Resposta

Através deste projeto, demonstramos que:

- **Técnicas de Pandas** (groupby, apply, lambda, filter) são essenciais para análise eficiente
- **Classificações baseadas em regras** permitem segmentar funcionários estrategicamente
- **Amostragem** ilustra a variabilidade natural dos dados

A criação da `Classificacao_Final` baseada em salário e engajamento oferece uma ferramenta estratégica para:

| Ação | Público-Alvo |
|------|-------------|
| Programas de reconhecimento | Excelente |
| Desenvolvimento e treinamento | Regular |
| Intervenção proativa | Atenção |

Este projeto evidencia como a análise de dados de RH fornece insights valiosos para tomada de decisão informada e proativa.

---

## Análise de Dados: Exercícios 24 e 25

### Exercício 24: Análise de Correlação entre Variáveis

**Objetivo:** Identificar relações entre variáveis numéricas do dataset.

In [ ]:
# Seleciona variáveis numéricas para correlação
variaveis_numericas = ['Salary', 'EngagementSurvey', 'EmpSatisfaction', 'Absences', 'DaysLateLast30', 'SpecialProjectsCount']

df[variaveis_numericas].corr()

In [ ]:
# Visualização da correlação
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.heatmap(df[variaveis_numericas].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlação entre Variáveis Numéricas', fontsize=14)
plt.tight_layout()
plt.show()

### Conclusões Exercício 24

| Correlação | Interpretação |
|------------|---------------|
| Salary × EngagementSurvey | Correlação fraca a moderada — salário mais alto não implica necessariamente maior engajamento |
| EmpSatisfaction × EngagementSurvey | Correlação positiva — funcionários mais satisfeitos tendem a estar mais engajados |
| Absences × DaysLateLast30 | Correlação moderada — funcionários ausentes também tendem a chegar atrasados |
| SpecialProjectsCount × Salary | Correlação fraca — projetos especiais não impactam diretamente o salário |

**Insight principal:** Engajamento e satisfação são mais correlacionados entre si do que com variáveis demográficas ou de حضور.

---

### Exercício 25: Análise de Distribuição e Outliers

**Objetivo:** Analisar a distribuição das variáveis e identificar possíveis outliers.

In [ ]:
# Análise descritiva do salário
print("Estatísticas do Salário:\n")
print(df['Salary'].describe())

In [ ]:
# Boxplot para identificar outliers
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['Salary'])
plt.title('Distribuição do Salário - Identificação de Outliers')
plt.xlabel('Salary')
plt.show()

In [ ]:
# Identifica outliers usando IQR
Q1 = df['Salary'].quantile(0.25)
Q3 = df['Salary'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df[(df['Salary'] < limite_inferior) | (df['Salary'] > limite_superior)]

print(f"Limite inferior: R$ {limite_inferior:,.2f}")
print(f"Limite superior: R$ {limite_superior:,.2f}")
print(f"\nTotal de outliers: {len(outliers)} funcionários")

print("\nOutliers identificados:")
outliers[['Employee_Name', 'Department', 'Position', 'Salary']].sort_values('Salary', ascending=False)

### Conclusões Exercício 25

1. **Distribuição Assimétrica:** A maioria dos salários está concentrada em uma faixa, com alguns valores extremely altos (outliers)

2. **Outliers Significativos:** 
   - Identificados funcionários com salários acima de R$ 165.000
   - Predominantemente do `Executive Office` e cargos de liderança

3. **Implicações:**
   - Para análises de média/mediana, considerar a presença de outliers
   - A mediana pode ser mais representativa que a média para salaries
   - Outliers podem representar executivos ou especialistas de alto valor

**Recomendação:** Para políticas de compensação, analisar outliers separadamente para entender se representam exceções ou erros nos dados.

---

*Instituto FIAP - Análise de Dados de Recursos Humanos*